In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Carga y análisis del dataset


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# 1. Carga desde Drive
ruta_maestra = "/content/drive/MyDrive/universal_filter_ngspice/datasets/sensor_dataset_v3.pt"
print(f"Cargando dataset maestro: {ruta_maestra}...")
data = torch.load(ruta_maestra, weights_only=False)

# 2. Análisis de Dimensiones
n_muestras = len(data['x'])
print(f"\n ANÁLISIS DEL DATASET")
print(f"{'-'*30}")
print(f" Total de muestras: {n_muestras}")

# 3. Análisis de Identidad (Circuit IDs)
ids = [m['circuit_id'] for m in data['x']]
conteo_ids = Counter(ids)
nombres_sensores = {0: "Térmico", 1: "Foto", 2: "Mecánico", 3: "Químico"}

print(" Distribución por Sensor:")
for cid, count in conteo_ids.items():
    nombre = nombres_sensores.get(cid, "Desconocido")
    print(f"   - {nombre} (ID {cid}): {count} muestras ({count/n_muestras*100:.1f}%)")

# 4. Análisis de Rangos (Knobs)
# Tomamos una muestra de cada para ver los rangos
print(" Rangos de Parámetros (Ejemplos):")
for cid in sorted(conteo_ids.keys()):
    idx = ids.index(cid)
    m = data['x'][idx]
    print(f"   - ID {cid}: {m['knob_names']} -> {m['knobs']}")

# 5. Verificación de Integridad (NaN/Inf)
# Revisamos una parte del tensor Y por velocidad
check_y = data['y'] if torch.is_tensor(data['y']) else torch.tensor(data['y'])
if torch.isnan(check_y).any() or torch.isinf(check_y).any():
    print(" ALERTA: Se detectaron valores NaN o Inf en los datos de salida.")
else:
    print(" Integridad de audio: OK (Sin NaN/Inf)")

In [ ]:
from torch.utils.data import DataLoader, random_split

class KaspixDataset(torch.utils.data.Dataset):
    def __init__(self, data_dict):
        self.x = data_dict['x']
        self.y = data_dict['y'] if torch.is_tensor(data_dict['y']) else torch.tensor(data_dict['y'])

        all_knobs = np.array([m['knobs'] for m in self.x])
        self.k_max = torch.tensor(all_knobs.max(axis=0)).float()
        self.k_min = torch.tensor(all_knobs.min(axis=0)).float()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        m = self.x[idx]

        # 1. Audio entrada
        audio_in = torch.tensor(m['audio_in']).float()
        if audio_in.dim() == 1: audio_in = audio_in.unsqueeze(0)

        # 2. NORMALIZACIÓN DE KNOBS (Evita los NaNs)
        raw_knobs = torch.tensor(m['knobs']).float()
        norm_knobs = (raw_knobs - self.k_min) / (self.k_max - self.k_min + 1e-9)

        # 3. Limpieza de Audio (Por si SPICE soltó algún valor loco)
        target = self.y[idx].float().unsqueeze(0)
        target = torch.nan_to_num(target, nan=0.0, posinf=1.0, neginf=-1.0)

        return {
            'audio_in': audio_in,
            'knobs': norm_knobs,
            'sensor_id': torch.tensor(m['circuit_id']).long(),
            'target': target
        }

# Crear el objeto Dataset
full_ds = KaspixDataset(data)

# Calcular tamaños
total = len(full_ds)
train_size = int(0.8 * total)
val_size = int(0.1 * total)
test_size = total - train_size - val_size

# Dividir aleatoriamente
train_ds, val_ds, test_ds = random_split(full_ds, [train_size, val_size, test_size])

# Crear DataLoaders
batch_size = 128
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

print(f" División completada:")
print(f"   - Entrenamiento: {train_size} muestras")
print(f"   - Validación:    {val_size} muestras")
print(f"   - Prueba (Test): {test_size} muestras")

In [ ]:
import torch
import torch.nn as nn

class FiLM(nn.Module):
    """Modula las características de la TCN basándose en los parámetros (Knobs + ID)"""
    def __init__(self, num_features, conditioning_dim):
        super().__init__()
        self.gain = nn.Linear(conditioning_dim, num_features)
        self.bias = nn.Linear(conditioning_dim, num_features)

    def forward(self, x, conditioning):
        # x: [Batch, Channels, Time]
        # conditioning: [Batch, conditioning_dim]
        g = self.gain(conditioning).unsqueeze(2)
        b = self.bias(conditioning).unsqueeze(2)
        return g * x + b

class TCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, conditioning_dim):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size,
                              padding=(kernel_size-1)*dilation//2, dilation=dilation)
        self.film = FiLM(out_channels, conditioning_dim)
        self.activation = nn.LeakyReLU(0.2)
        self.resample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, cond):
        res = self.resample(x)
        x = self.conv(x)
        x = self.film(x, cond)
        x = self.activation(x)
        return x + res

class KaspixUniversalTCN(nn.Module):
    def __init__(self, num_sensors=4, conditioning_dim=16):
        super().__init__()

        # 1. Procesamiento de Condicionamiento (ID + Knobs)
        self.sensor_embedding = nn.Embedding(num_sensors, 8)
        self.cond_mlp = nn.Sequential(
            nn.Linear(8 + 2, conditioning_dim),
            nn.ReLU(),
            nn.Linear(conditioning_dim, conditioning_dim)
        )

        # 2. Backbone de la TCN
        self.layers = nn.ModuleList([
            TCNBlock(1, 32, kernel_size=13, dilation=1, conditioning_dim=conditioning_dim),
            TCNBlock(32, 32, kernel_size=13, dilation=2, conditioning_dim=conditioning_dim),
            TCNBlock(32, 64, kernel_size=13, dilation=4, conditioning_dim=conditioning_dim),
            TCNBlock(64, 64, kernel_size=13, dilation=8, conditioning_dim=conditioning_dim),
            ])
        self.output_conv = nn.Conv1d(64, 1, kernel_size=1)

    def forward(self, audio, knobs, sensor_id):
        emb = self.sensor_embedding(sensor_id)
        cond = torch.cat([emb, knobs], dim=1)
        cond = self.cond_mlp(cond)

        # Pasar por la TCN
        x = audio
        for layer in self.layers:
            x = layer(x, cond)

        return self.output_conv(x)

# Instanciar modelo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = KaspixUniversalTCN().to(device)
print(f" Modelo listo en {device}")

In [ ]:
import torch
import torch.nn as nn

class SensorConditioning(nn.Module):
    """Capa compartida para procesar el ID del sensor y los Knobs"""
    def __init__(self, num_sensors, conditioning_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_sensors, 8)
        self.mlp = nn.Sequential(
            nn.Linear(8 + 2, conditioning_dim),
            nn.ReLU(),
            nn.Linear(conditioning_dim, conditioning_dim)
        )
    def forward(self, sensor_id, knobs):
        emb = self.embedding(sensor_id)
        cond = torch.cat([emb, knobs], dim=1)
        return self.mlp(cond)

# --- MODELO RNN ---
class KaspixRNN(nn.Module):
    def __init__(self, hidden_dim=64, cond_dim=16):
        super().__init__()
        self.cond_proc = SensorConditioning(4, cond_dim)
        # RNN que procesa el audio muestra por muestra (o bloques)
        self.rnn = nn.RNN(input_size=1 + cond_dim, hidden_size=hidden_dim,
                          num_layers=2, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, audio, knobs, sensor_id):
        # audio: [B, 1, T] -> [B, T, 1]
        x = audio.transpose(1, 2)
        cond = self.cond_proc(sensor_id, knobs) # [B, cond_dim]

        # Expandimos el condicionamiento para cada paso de tiempo
        cond_expanded = cond.unsqueeze(1).expand(-1, x.size(1), -1)
        x = torch.cat([x, cond_expanded], dim=-1) # [B, T, 1 + cond_dim]

        out, _ = self.rnn(x)
        out = self.fc(out)
        return out.transpose(1, 2) # [B, 1, T]

# --- MODELO LSTM ---
class KaspixLSTM(nn.Module):
    def __init__(self, hidden_dim=64, cond_dim=16):
        super().__init__()
        self.cond_proc = SensorConditioning(4, cond_dim)
        self.lstm = nn.LSTM(input_size=1 + cond_dim, hidden_size=hidden_dim,
                            num_layers=2, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, audio, knobs, sensor_id):
        x = audio.transpose(1, 2)
        cond = self.cond_proc(sensor_id, knobs)

        cond_expanded = cond.unsqueeze(1).expand(-1, x.size(1), -1)
        x = torch.cat([x, cond_expanded], dim=-1)

        out, _ = self.lstm(x)
        out = self.fc(out)
        return out.transpose(1, 2)

TCN

In [ ]:
import torch.optim as optim
import time
import os
from tqdm.auto import tqdm

# --- 1. LAS 3 MÉTRICAS ---
def compute_metrics(pred, target):
    # a. ESR (Error to Signal Ratio)
    noise = torch.sum((target - pred) ** 2)
    signal = torch.sum(target ** 2) + 1e-7
    esr = noise / signal

    # b. L1 (Mean Absolute Error)
    l1 = torch.mean(torch.abs(target - pred))

    # c. Correlation (Pearson-like)
    target_std = target - torch.mean(target)
    pred_std = pred - torch.mean(pred)
    corr = torch.sum(target_std * pred_std) / (torch.sqrt(torch.sum(target_std**2) * torch.sum(pred_std**2)) + 1e-7)

    return esr.item(), l1.item(), corr.item()

# --- 2. PÉRDIDA COMBINADA ---
class KaspixLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.L1Loss()

    def forward(self, pred, target):
        # El modelo optimiza principalmente para bajar el ESR y el L1
        esr = torch.sum((target - pred) ** 2) / (torch.sum(target ** 2) + 1e-7)
        return 0.7 * esr + 0.3 * self.l1(pred, target)

# --- 3. CONFIGURACIÓN E INICIO ---
MODEL_TYPE = "TCN" # Puedes cambiar a "LSTM" o "RNN"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if MODEL_TYPE == "TCN":
    model = KaspixUniversalTCN().to(device)
elif MODEL_TYPE == "LSTM":
    model = KaspixLSTM(hidden_dim=64).to(device)
elif MODEL_TYPE == "RNN":
    model = KaspixRNN(hidden_dim=64).to(device)
print(f" Modelo seleccionado: {MODEL_TYPE} en {device}")

# (Aquí asumimos que ya instanciaste el 'model' según el MODEL_TYPE anterior)
criterion = KaspixLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

SAVE_DIR = "/content/drive/MyDrive/universal_filter_ngspice/checkpoints/sensores"
os.makedirs(SAVE_DIR, exist_ok=True) # Crea la carpeta si no existe

history = {'train_loss': [], 'val_loss': [], 'val_esr': [], 'val_l1': [], 'val_corr': []}
best_val_loss = float('inf')
epochs = 30
patience_es = 10
counter_es = 0

print(f" Entrenando {MODEL_TYPE} con triple métrica...")

for epoch in range(epochs):
    model.train()
    t_losses = []
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]', leave=False)
    for batch in pbar:
        # Extraer datos
        audio_in, knobs = batch['audio_in'].to(device), batch['knobs'].to(device)
        s_id, target = batch['sensor_id'].to(device), batch['target'].to(device)

        optimizer.zero_grad()
        out = model(audio_in, knobs, s_id)
        loss = criterion(out, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()
        t_losses.append(loss.item())
        pbar.set_postfix({'Loss': f'{loss.item():.6f}'})

    # Evaluación con las 3 métricas
    model.eval()
    v_losses, v_esrs, v_l1s, v_corrs = [], [], [], []
    pbar_val = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)
    with torch.no_grad():
        for batch in pbar_val:
            audio_in, knobs = batch['audio_in'].to(device), batch['knobs'].to(device)
            s_id, target = batch['sensor_id'].to(device), batch['target'].to(device)

            out = model(audio_in, knobs, s_id)
            v_losses.append(criterion(out, target).item())

            esr, l1, corr = compute_metrics(out, target)
            v_esrs.append(esr)
            v_l1s.append(l1)
            v_corrs.append(corr)
            pbar_val.set_postfix({'esr': f'{esr:-6f}'})

    # Consolidar resultados de la época
    epoch_val_loss = sum(v_losses)/len(v_losses)
    history['train_loss'].append(sum(t_losses)/len(t_losses))
    history['val_loss'].append(epoch_val_loss)
    history['val_esr'].append(sum(v_esrs)/len(v_esrs))
    history['val_corr'].append(sum(v_corrs)/len(v_corrs))

    print(f"Ep {epoch+1}: Loss {epoch_val_loss:.6f} | ESR {history['val_esr'][-1]:.6f} | Corr {history['val_corr'][-1]:.6f}")

    scheduler.step(epoch_val_loss)

    # Guardar el mejor
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        checkpoint_path = os.path.join(SAVE_DIR, f"best_tcn.pth")
        torch.save(model.state_dict(), checkpoint_path)
        counter_es = 0
    else:
        counter_es += 1
        if counter_es >= patience_es: break

LSTM

In [ ]:
import torch.optim as optim
import time
import os
from tqdm.auto import tqdm

# 1. LAS 3 MÉTRICAS
def compute_metrics(pred, target):
    # a. ESR (Error to Signal Ratio)
    noise = torch.sum((target - pred) ** 2)
    signal = torch.sum(target ** 2) + 1e-7
    esr = noise / signal

    # b. L1 (Mean Absolute Error)
    l1 = torch.mean(torch.abs(target - pred))

    # c. Correlation
    target_std = target - torch.mean(target)
    pred_std = pred - torch.mean(pred)
    corr = torch.sum(target_std * pred_std) / (torch.sqrt(torch.sum(target_std**2) * torch.sum(pred_std**2)) + 1e-7)

    return esr.item(), l1.item(), corr.item()

# 2. PÉRDIDA COMBINADA
class KaspixLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.L1Loss()

    def forward(self, pred, target):
        # El modelo optimiza principalmente para bajar el ESR y el L1
        esr = torch.sum((target - pred) ** 2) / (torch.sum(target ** 2) + 1e-7)
        return 0.7 * esr + 0.3 * self.l1(pred, target)

# 3. CONFIGURACIÓN E INICIO
MODEL_TYPE = "LSTM" # Cambiar a "LSTM" o "RNN"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if MODEL_TYPE == "TCN":
    model = KaspixUniversalTCN().to(device)
elif MODEL_TYPE == "LSTM":
    model_lstm = KaspixLSTM(hidden_dim=64).to(device)
elif MODEL_TYPE == "RNN":
    model = KaspixRNN(hidden_dim=64).to(device)
print(f" Modelo seleccionado: {MODEL_TYPE} en {device}")

criterion = KaspixLoss()
optimizer = optim.Adam(model_lstm.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

SAVE_DIR = "/content/drive/MyDrive/universal_filter_ngspice/checkpoints/sensores"
os.makedirs(SAVE_DIR, exist_ok=True) # Crea la carpeta

history = {'train_loss': [], 'val_loss': [], 'val_esr': [], 'val_l1': [], 'val_corr': []}
best_val_loss = float('inf')
epochs = 30
patience_es = 10
counter_es = 0

print(f" Entrenando {MODEL_TYPE} con triple métrica...")

for epoch in range(epochs):
    model_lstm.train()
    t_losses = []
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]', leave=False)
    for batch in pbar:
        # Extraer datos
        audio_in, knobs = batch['audio_in'].to(device), batch['knobs'].to(device)
        s_id, target = batch['sensor_id'].to(device), batch['target'].to(device)

        optimizer.zero_grad()
        out = model_lstm(audio_in, knobs, s_id)
        loss = criterion(out, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_lstm.parameters(), max_norm=1)
        optimizer.step()
        t_losses.append(loss.item())
        pbar.set_postfix({'Loss': f'{loss.item():.6f}'})

    # Evaluación con las 3 métricas
    model_lstm.eval()
    v_losses, v_esrs, v_l1s, v_corrs = [], [], [], []
    pbar_val = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)
    with torch.no_grad():
        for batch in pbar_val:
            audio_in, knobs = batch['audio_in'].to(device), batch['knobs'].to(device)
            s_id, target = batch['sensor_id'].to(device), batch['target'].to(device)

            out = model_lstm(audio_in, knobs, s_id)
            v_losses.append(criterion(out, target).item())

            esr, l1, corr = compute_metrics(out, target)
            v_esrs.append(esr)
            v_l1s.append(l1)
            v_corrs.append(corr)
            pbar_val.set_postfix({'esr': f'{esr:-6f}'})

    # Consolidar resultados de la época
    epoch_val_loss = sum(v_losses)/len(v_losses)
    history['train_loss'].append(sum(t_losses)/len(t_losses))
    history['val_loss'].append(epoch_val_loss)
    history['val_esr'].append(sum(v_esrs)/len(v_esrs))
    history['val_corr'].append(sum(v_corrs)/len(v_corrs))

    print(f"Ep {epoch+1}: Loss {epoch_val_loss:.6f} | ESR {history['val_esr'][-1]:.6f} | Corr {history['val_corr'][-1]:.6f}")

    scheduler.step(epoch_val_loss)

    # Guardar el mejor
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        checkpoint_path = os.path.join(SAVE_DIR, f"best_lstm.pth")
        torch.save(model_lstm.state_dict(), checkpoint_path)
        counter_es = 0
    else:
        counter_es += 1
        if counter_es >= patience_es: break

RNN

In [ ]:
import torch.optim as optim
import time
import os
from tqdm.auto import tqdm

# --- 1. LAS 3 MÉTRICAS ---
def compute_metrics(pred, target):
    # a. ESR (Error to Signal Ratio)
    noise = torch.sum((target - pred) ** 2)
    signal = torch.sum(target ** 2) + 1e-7
    esr = noise / signal

    # b. L1 (Mean Absolute Error)
    l1 = torch.mean(torch.abs(target - pred))

    # c. Correlation (Pearson-like)
    target_std = target - torch.mean(target)
    pred_std = pred - torch.mean(pred)
    corr = torch.sum(target_std * pred_std) / (torch.sqrt(torch.sum(target_std**2) * torch.sum(pred_std**2)) + 1e-7)

    return esr.item(), l1.item(), corr.item()

# --- 2. PÉRDIDA COMBINADA ---
class KaspixLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.L1Loss()

    def forward(self, pred, target):
        # El modelo optimiza principalmente para bajar el ESR y el L1
        esr = torch.sum((target - pred) ** 2) / (torch.sum(target ** 2) + 1e-7)
        return 0.7 * esr + 0.3 * self.l1(pred, target)

# --- 3. CONFIGURACIÓN E INICIO ---
MODEL_TYPE = "RNN" # Puedes cambiar a "LSTM" o "RNN"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if MODEL_TYPE == "TCN":
    model = KaspixUniversalTCN().to(device)
elif MODEL_TYPE == "LSTM":
    model = KaspixLSTM(hidden_dim=64).to(device)
elif MODEL_TYPE == "RNN":
    model_rnn = KaspixRNN(hidden_dim=64).to(device)
print(f" Modelo seleccionado: {MODEL_TYPE} en {device}")

# (Aquí asumimos que ya instanciaste el 'model' según el MODEL_TYPE anterior)
criterion = KaspixLoss()
optimizer = optim.Adam(model_rnn.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

SAVE_DIR = "/content/drive/MyDrive/universal_filter_ngspice/checkpoints/sensores"
os.makedirs(SAVE_DIR, exist_ok=True) # Crea la carpeta si no existe

history = {'train_loss': [], 'val_loss': [], 'val_esr': [], 'val_l1': [], 'val_corr': []}
best_val_loss = float('inf')
epochs = 30
patience_es = 10
counter_es = 0

print(f" Entrenando {MODEL_TYPE} con triple métrica...")

for epoch in range(epochs):
    model_rnn.train()
    t_losses = []
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]', leave=False)
    for batch in pbar:
        # Extraer datos
        audio_in, knobs = batch['audio_in'].to(device), batch['knobs'].to(device)
        s_id, target = batch['sensor_id'].to(device), batch['target'].to(device)

        optimizer.zero_grad()
        out = model_rnn(audio_in, knobs, s_id)
        loss = criterion(out, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_rnn.parameters(), max_norm=1)
        optimizer.step()
        t_losses.append(loss.item())
        pbar.set_postfix({'Loss': f'{loss.item():.6f}'})

    # Evaluación con las 3 métricas
    model_rnn.eval()
    v_losses, v_esrs, v_l1s, v_corrs = [], [], [], []
    pbar_val = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)
    with torch.no_grad():
        for batch in pbar_val:
            audio_in, knobs = batch['audio_in'].to(device), batch['knobs'].to(device)
            s_id, target = batch['sensor_id'].to(device), batch['target'].to(device)

            out = model_rnn(audio_in, knobs, s_id)
            v_losses.append(criterion(out, target).item())

            esr, l1, corr = compute_metrics(out, target)
            v_esrs.append(esr)
            v_l1s.append(l1)
            v_corrs.append(corr)
            pbar_val.set_postfix({'esr': f'{esr:-6f}'})

    # Consolidar resultados de la época
    epoch_val_loss = sum(v_losses)/len(v_losses)
    history['train_loss'].append(sum(t_losses)/len(t_losses))
    history['val_loss'].append(epoch_val_loss)
    history['val_esr'].append(sum(v_esrs)/len(v_esrs))
    history['val_corr'].append(sum(v_corrs)/len(v_corrs))

    print(f"Ep {epoch+1}: Loss {epoch_val_loss:.6f} | ESR {history['val_esr'][-1]:.6f} | Corr {history['val_corr'][-1]:.6f}")

    scheduler.step(epoch_val_loss)

    # Guardar el mejor
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        checkpoint_path = os.path.join(SAVE_DIR, f"best_rnn.pth")
        torch.save(model_rnn.state_dict(), checkpoint_path)
        counter_es = 0
    else:
        counter_es += 1
        if counter_es >= patience_es: break

In [ ]:
import torch
import os

# 1. Definir la ruta de Drive donde están tus checkpoints
DRIVE_PATH = "/content/drive/MyDrive/universal_filter_ngspice/checkpoints/sensores"

def cargar_modelo(modelo_clase, nombre_archivo, device, **kwargs):
    # Construimos la ruta completa
    full_path = os.path.join(DRIVE_PATH, nombre_archivo)

    if not os.path.exists(full_path):
        raise FileNotFoundError(f"No se encontró el archivo en: {full_path}")

    # Instanciamos la clase
    model = modelo_clase(**kwargs).to(device)
    # Cargamos los pesos guardados
    model.load_state_dict(torch.load(full_path, map_location=device))
    model.eval()
    return model

# 2. Diccionario de modelos cargados
dict_modelos = {}

print(f" Buscando modelos en: {DRIVE_PATH}\n")

# Intentar cargar TCN
try:
    dict_modelos['TCN'] = cargar_modelo(KaspixUniversalTCN, 'best_tcn.pth', device)
    print(" TCN loaded successfully from Drive")
except Exception as e:
    print(f" TCN not loaded: {e}")

# Intentar cargar LSTM
try:
    dict_modelos['LSTM'] = cargar_modelo(KaspixLSTM, 'best_lstm.pth', device, hidden_dim=64)
    print(" LSTM loaded successfully from Drive")
except Exception as e:
    print(f" LSTM not loaded: {e}")

# Intentar cargar RNN
try:
    dict_modelos['RNN'] = cargar_modelo(KaspixRNN, 'best_rnn.pth', device, hidden_dim=64)
    print(" RNN loaded successfully from Drive")
except Exception as e:
    print(f" RNN not loaded: {e}")

Gráficos TCN

In [ ]:
# Visualización con error desde best

def visualize_prediction(model, dataset, device, idx=None):
    model.eval()
    if idx is None:
        idx = np.random.randint(len(dataset))

    # 1. Get data from dataset
    item = dataset[idx]
    audio_in = item['audio_in'].unsqueeze(0).to(device)
    knobs = item['knobs'].unsqueeze(0).to(device)
    s_id = item['sensor_id'].unsqueeze(0).to(device)
    target = item['target'].squeeze().cpu().numpy() # Ensure it's on CPU for numpy

    # 2. Inference
    with torch.no_grad():
        pred = model(audio_in, knobs, s_id).cpu().squeeze().numpy()

    # English sensor mapping
    sensor_names = {0: "Thermal", 1: "Photo", 2: "Mechanical", 3: "Chemical"}
    sid_val = item['sensor_id'].item()
    sensor_name = sensor_names.get(sid_val, "Unknown")

    # 3. Plotting
    plt.figure(figsize=(15, 8))

    # Subplot 1: Waveform Comparison
    plt.subplot(2, 1, 1)
    plt.plot(target, label='SPICE Target', alpha=0.6, color='blue', lw=2)
    plt.plot(pred, label='TCN Prediction', linestyle='--', color='orange', lw=1.5)

    plt.title(f"Waveform Comparison - {sensor_name} Sensor (ID: {sid_val}) | Knobs: {item['knobs'].numpy()}", fontsize=14)
    plt.xlabel("Time Samples")
    plt.ylabel("Normalized Amplitude")
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)

    # Subplot 2: Residual Error
    plt.subplot(2, 1, 2)
    error = target - pred
    plt.fill_between(range(len(error)), error, color='red', alpha=0.3, label='Error Area')
    plt.plot(error, color='red', lw=0.6)

    plt.title(f"Residual Error (Mean Absolute Error: {np.abs(error).mean():.6f})", fontsize=12)
    plt.xlabel("Time Samples")
    plt.ylabel("Error Magnitude")
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# --- EXECUTION ---
# Usamos el modelo cargado desde el diccionario de Drive
if 'TCN' in dict_modelos:
    visualize_prediction(dict_modelos['TCN'], test_ds, device)
else:
    print(" The TCN model was not found in dict_modelos. Please check the Drive path.")

In [ ]:
# Respuesta en frecuencia

import torch
import numpy as np
import matplotlib.pyplot as plt

def plot_comparative_bode(model, dataset, device, model_name="Model", idx=None, fs=48000):
    model.eval()
    if idx is None:
        idx = np.random.randint(len(dataset))

    # 1. Get data from dataset
    item = dataset[idx]
    x_audio = item['audio_in'].unsqueeze(0).to(device) # Input (Stimulus)
    y_real = item['target'].squeeze().cpu().numpy()    # SPICE Output
    knobs = item['knobs'].unsqueeze(0).to(device)
    s_id = item['sensor_id'].unsqueeze(0).to(device)

    # 2. Get Model Prediction
    with torch.no_grad():
        y_pred = model(x_audio, knobs, s_id)

        # Ajuste de dimensiones para modelos recurrentes (LSTM/RNN)
        # Si devuelve [B, T, 1] lo pasamos a [T]
        y_pred = y_pred.cpu().squeeze().numpy()
        if y_pred.ndim > 1: y_pred = y_pred[0]

    x_audio_np = x_audio.cpu().squeeze().numpy()

    # 3. Calculate Transfer Functions (H = FFT(output) / FFT(input))
    n_fft = 2048
    X = np.fft.rfft(x_audio_np, n=n_fft)
    Y_real = np.fft.rfft(y_real, n=n_fft)
    Y_pred = np.fft.rfft(y_pred, n=n_fft)

    # Avoid division by zero
    H_real = Y_real / (X + 1e-9)
    H_pred = Y_pred / (X + 1e-9)

    freqs = np.fft.rfftfreq(n_fft, d=1/fs)

    # 4. Plotting
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

    # Sensor mapping for the title
    sensor_names = {0: "Thermal", 1: "Photo", 2: "Mechanical", 3: "Chemical"}
    sid_val = item['sensor_id'].item()
    sensor_name = sensor_names.get(sid_val, "Unknown")

    # Magnitude Plot
    ax1.semilogx(freqs, 20*np.log10(np.abs(H_real) + 1e-9), label='SPICE Target', color='black', lw=2, alpha=0.6)
    ax1.semilogx(freqs, 20*np.log10(np.abs(H_pred) + 1e-9), label=f'TCN Prediction', color='blue', linestyle='--', lw=2)
    ax1.set_ylabel('Magnitude (dB)', fontsize=12)
    ax1.set_title(f'Frequency Response Comparison - {sensor_name} Sensor (ID: {sid_val})', fontsize=14)
    ax1.legend()
    ax1.grid(True, which="both", alpha=0.3)

    # Phase Plot
    ax2.semilogx(freqs, np.angle(H_real, deg=True), color='black', lw=2, alpha=0.6)
    ax2.semilogx(freqs, np.angle(H_pred, deg=True), color='blue', linestyle='--', lw=2)
    ax2.set_ylabel('Phase (Degrees)', fontsize=12)
    ax2.set_xlabel('Frequency (Hz)', fontsize=12)
    ax2.grid(True, which="both", alpha=0.3)

    plt.tight_layout()
    plt.show()

if 'TCN' in dict_modelos:
    plot_comparative_bode(dict_modelos['TCN'], test_ds, device, model_name="LSTM")
else:
    print(" TCN model not found in dict_modelos.")

In [ ]:
# Respuesta de sensores

def plot_universal_results(model, test_loader, device, num_samples=4):
    model.eval()
    samples_found = {}
    # Translated sensor names for the report
    sensor_names_en = {0: "Thermal", 1: "Photo", 2: "Mechanical", 3: "Chemical"}

    # 1. Search for one example of each sensor type in the test set
    with torch.no_grad():
        for batch in test_loader:
            audio_in = batch['audio_in'].to(device)
            knobs = batch['knobs'].to(device)
            s_ids = batch['sensor_id']
            targets = batch['target'].to(device)

            # Inference
            preds = model(audio_in, knobs, s_ids.to(device))

            # Ensure predictions match target dimensions [B, 1, T]
            if preds.shape != targets.shape:
                preds = preds.transpose(1, 2) if preds.shape[1] != 1 else preds

            for i in range(len(s_ids)):
                sid = s_ids[i].item()
                if sid not in samples_found and len(samples_found) < num_samples:
                    samples_found[sid] = {
                        'target': targets[i].cpu().squeeze().numpy(),
                        'pred': preds[i].cpu().squeeze().numpy(),
                        'knobs': batch['knobs'][i].numpy(),
                        'name': sensor_names_en.get(sid, f"ID {sid}")
                    }
            if len(samples_found) == num_samples: break

    # 2. Plotting
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.flatten()

    for i, (sid, data) in enumerate(sorted(samples_found.items())):
        ax = axes[i]
        t = data['target']
        p = data['pred']

        # Local metrics for the plot
        noise = np.sum((t - p)**2)
        signal = np.sum(t**2) + 1e-7
        esr = noise / signal
        corr = np.corrcoef(t, p)[0, 1]

        # Plot Ground Truth vs Prediction
        ax.plot(t, label='SPICE (Ground Truth)', color='black', alpha=0.4, lw=2)
        ax.plot(p, label='TCN Prediction', color='red', linestyle='--', lw=1.5)

        ax.set_title(f"Sensor: {data['name']} (ID: {sid})\nESR: {esr:.6f} | Corr: {corr:.4f}", fontsize=12)
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_xlabel("Time Samples")
        ax.set_ylabel("Normalized Amplitude (V)")

    plt.suptitle("TCN Universal Model Performance Across All Sensors", fontsize=16, y=1.02)
    plt.tight_layout()

    # Save figure
    save_path = "universal_tcn_results_en.png"
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    print(f" Comparison plot saved as '{save_path}'")

# --- EXECUTION ---
# Using the TCN model loaded from your Drive dictionary
if 'TCN' in dict_modelos:
    plot_universal_results(dict_modelos['TCN'], test_loader, device)
else:
    print(" Error: 'TCN' model not found in dict_modelos. Please verify the Drive loading step.")

Gráficos LSTM

In [ ]:
# Visualización con error desde best

def visualize_prediction(model, dataset, device, idx=None):
    model.eval()
    if idx is None:
        idx = np.random.randint(len(dataset))

    # 1. Get data from dataset
    item = dataset[idx]
    audio_in = item['audio_in'].unsqueeze(0).to(device)
    knobs = item['knobs'].unsqueeze(0).to(device)
    s_id = item['sensor_id'].unsqueeze(0).to(device)
    target = item['target'].squeeze().cpu().numpy() # Ensure it's on CPU for numpy

    # 2. Inference
    with torch.no_grad():
        pred = model(audio_in, knobs, s_id).cpu().squeeze().numpy()

    # English sensor mapping
    sensor_names = {0: "Thermal", 1: "Photo", 2: "Mechanical", 3: "Chemical"}
    sid_val = item['sensor_id'].item()
    sensor_name = sensor_names.get(sid_val, "Unknown")

    # 3. Plotting
    plt.figure(figsize=(15, 8))

    # Subplot 1: Waveform Comparison
    plt.subplot(2, 1, 1)
    plt.plot(target, label='SPICE Target', alpha=0.6, color='blue', lw=2)
    plt.plot(pred, label='LSTM Prediction', linestyle='--', color='orange', lw=1.5)

    plt.title(f"Waveform Comparison - {sensor_name} Sensor (ID: {sid_val}) | Knobs: {item['knobs'].numpy()}", fontsize=14)
    plt.xlabel("Time Samples")
    plt.ylabel("Normalized Amplitude")
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)

    # Subplot 2: Residual Error
    plt.subplot(2, 1, 2)
    error = target - pred
    plt.fill_between(range(len(error)), error, color='red', alpha=0.3, label='Error Area')
    plt.plot(error, color='red', lw=0.6)

    plt.title(f"Residual Error (Mean Absolute Error: {np.abs(error).mean():.6f})", fontsize=12)
    plt.xlabel("Time Samples")
    plt.ylabel("Error Magnitude")
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# --- EXECUTION ---
# Usamos el modelo cargado desde el diccionario de Drive
if 'LSTM' in dict_modelos:
    visualize_prediction(dict_modelos['LSTM'], test_ds, device)
else:
    print(" The LSTM model was not found in dict_modelos. Please check the Drive path.")

In [ ]:
# Respuesta en frecuencia

import torch
import numpy as np
import matplotlib.pyplot as plt

def plot_comparative_bode(model, dataset, device, model_name="Model", idx=None, fs=48000):
    model.eval()
    if idx is None:
        idx = np.random.randint(len(dataset))

    # 1. Get data from dataset
    item = dataset[idx]
    x_audio = item['audio_in'].unsqueeze(0).to(device) # Input (Stimulus)
    y_real = item['target'].squeeze().cpu().numpy()    # SPICE Output
    knobs = item['knobs'].unsqueeze(0).to(device)
    s_id = item['sensor_id'].unsqueeze(0).to(device)

    # 2. Get Model Prediction
    with torch.no_grad():
        y_pred = model(x_audio, knobs, s_id)

        # Ajuste de dimensiones para modelos recurrentes (LSTM/RNN)
        # Si devuelve [B, T, 1] lo pasamos a [T]
        y_pred = y_pred.cpu().squeeze().numpy()
        if y_pred.ndim > 1: y_pred = y_pred[0]

    x_audio_np = x_audio.cpu().squeeze().numpy()

    # 3. Calculate Transfer Functions (H = FFT(output) / FFT(input))
    n_fft = 2048
    X = np.fft.rfft(x_audio_np, n=n_fft)
    Y_real = np.fft.rfft(y_real, n=n_fft)
    Y_pred = np.fft.rfft(y_pred, n=n_fft)

    # Avoid division by zero
    H_real = Y_real / (X + 1e-9)
    H_pred = Y_pred / (X + 1e-9)

    freqs = np.fft.rfftfreq(n_fft, d=1/fs)

    # 4. Plotting
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

    # Sensor mapping for the title
    sensor_names = {0: "Thermal", 1: "Photo", 2: "Mechanical", 3: "Chemical"}
    sid_val = item['sensor_id'].item()
    sensor_name = sensor_names.get(sid_val, "Unknown")

    # Magnitude Plot
    ax1.semilogx(freqs, 20*np.log10(np.abs(H_real) + 1e-9), label='SPICE Target', color='black', lw=2, alpha=0.6)
    ax1.semilogx(freqs, 20*np.log10(np.abs(H_pred) + 1e-9), label=f'LSTM Prediction', color='blue', linestyle='--', lw=2)
    ax1.set_ylabel('Magnitude (dB)', fontsize=12)
    ax1.set_title(f'Frequency Response Comparison - {sensor_name} Sensor (ID: {sid_val})', fontsize=14)
    ax1.legend()
    ax1.grid(True, which="both", alpha=0.3)

    # Phase Plot
    ax2.semilogx(freqs, np.angle(H_real, deg=True), color='black', lw=2, alpha=0.6)
    ax2.semilogx(freqs, np.angle(H_pred, deg=True), color='blue', linestyle='--', lw=2)
    ax2.set_ylabel('Phase (Degrees)', fontsize=12)
    ax2.set_xlabel('Frequency (Hz)', fontsize=12)
    ax2.grid(True, which="both", alpha=0.3)

    plt.tight_layout()
    plt.show()

# --- EJECUCIÓN ---
# Usamos el modelo LSTM cargado desde Drive
if 'LSTM' in dict_modelos:
    plot_comparative_bode(dict_modelos['LSTM'], test_ds, device, model_name="LSTM")
else:
    print(" LSTM model not found in dict_modelos.")

In [ ]:
# Respuesta de sensores

def plot_universal_results(model, test_loader, device, num_samples=4):
    model.eval()
    samples_found = {}
    # Translated sensor names for the report
    sensor_names_en = {0: "Thermal", 1: "Photo", 2: "Mechanical", 3: "Chemical"}

    # 1. Search for one example of each sensor type in the test set
    with torch.no_grad():
        for batch in test_loader:
            audio_in = batch['audio_in'].to(device)
            knobs = batch['knobs'].to(device)
            s_ids = batch['sensor_id']
            targets = batch['target'].to(device)

            # Inference
            preds = model(audio_in, knobs, s_ids.to(device))

            # Ensure predictions match target dimensions [B, 1, T]
            if preds.shape != targets.shape:
                preds = preds.transpose(1, 2) if preds.shape[1] != 1 else preds

            for i in range(len(s_ids)):
                sid = s_ids[i].item()
                if sid not in samples_found and len(samples_found) < num_samples:
                    samples_found[sid] = {
                        'target': targets[i].cpu().squeeze().numpy(),
                        'pred': preds[i].cpu().squeeze().numpy(),
                        'knobs': batch['knobs'][i].numpy(),
                        'name': sensor_names_en.get(sid, f"ID {sid}")
                    }
            if len(samples_found) == num_samples: break

    # 2. Plotting
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.flatten()

    for i, (sid, data) in enumerate(sorted(samples_found.items())):
        ax = axes[i]
        t = data['target']
        p = data['pred']

        # Local metrics for the plot
        noise = np.sum((t - p)**2)
        signal = np.sum(t**2) + 1e-7
        esr = noise / signal
        corr = np.corrcoef(t, p)[0, 1]

        # Plot Ground Truth vs Prediction
        ax.plot(t, label='SPICE (Ground Truth)', color='black', alpha=0.4, lw=2)
        ax.plot(p, label='LSTM Prediction', color='red', linestyle='--', lw=1.5)

        ax.set_title(f"Sensor: {data['name']} (ID: {sid})\nESR: {esr:.6f} | Corr: {corr:.4f}", fontsize=12)
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_xlabel("Time Samples")
        ax.set_ylabel("Normalized Amplitude (V)")

    plt.suptitle("LSTM Universal Model Performance Across All Sensors", fontsize=16, y=1.02)
    plt.tight_layout()

    # Save figure
    save_path = "universal_lstm_results_en.png"
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    print(f" Comparison plot saved as '{save_path}'")

# --- EXECUTION ---
# Using the TCN model loaded from your Drive dictionary
if 'LSTM' in dict_modelos:
    plot_universal_results(dict_modelos['LSTM'], test_loader, device)
else:
    print(" Error: 'LSTM' model not found in dict_modelos. Please verify the Drive loading step.")

Gráficos RNN

In [ ]:
# Visualización con error desde best

def visualize_prediction(model, dataset, device, idx=None):
    model.eval()
    if idx is None:
        idx = np.random.randint(len(dataset))

    # 1. Get data from dataset
    item = dataset[idx]
    audio_in = item['audio_in'].unsqueeze(0).to(device)
    knobs = item['knobs'].unsqueeze(0).to(device)
    s_id = item['sensor_id'].unsqueeze(0).to(device)
    target = item['target'].squeeze().cpu().numpy() # Ensure it's on CPU for numpy

    # 2. Inference
    with torch.no_grad():
        pred = model(audio_in, knobs, s_id).cpu().squeeze().numpy()

    # English sensor mapping
    sensor_names = {0: "Thermal", 1: "Photo", 2: "Mechanical", 3: "Chemical"}
    sid_val = item['sensor_id'].item()
    sensor_name = sensor_names.get(sid_val, "Unknown")

    # 3. Plotting
    plt.figure(figsize=(15, 8))

    # Subplot 1: Waveform Comparison
    plt.subplot(2, 1, 1)
    plt.plot(target, label='SPICE Target', alpha=0.6, color='blue', lw=2)
    plt.plot(pred, label='RNN Prediction', linestyle='--', color='orange', lw=1.5)

    plt.title(f"Waveform Comparison - {sensor_name} Sensor (ID: {sid_val}) | Knobs: {item['knobs'].numpy()}", fontsize=14)
    plt.xlabel("Time Samples")
    plt.ylabel("Normalized Amplitude")
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)

    # Subplot 2: Residual Error
    plt.subplot(2, 1, 2)
    error = target - pred
    plt.fill_between(range(len(error)), error, color='red', alpha=0.3, label='Error Area')
    plt.plot(error, color='red', lw=0.6)

    plt.title(f"Residual Error (Mean Absolute Error: {np.abs(error).mean():.6f})", fontsize=12)
    plt.xlabel("Time Samples")
    plt.ylabel("Error Magnitude")
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# --- EXECUTION ---
# Usamos el modelo cargado desde el diccionario de Drive
if 'RNN' in dict_modelos:
    visualize_prediction(dict_modelos['RNN'], test_ds, device)
else:
    print(" The RNN model was not found in dict_modelos. Please check the Drive path.")

In [ ]:
# Respuesta en frecuencia

import torch
import numpy as np
import matplotlib.pyplot as plt

def plot_comparative_bode(model, dataset, device, model_name="Model", idx=None, fs=48000):
    model.eval()
    if idx is None:
        idx = np.random.randint(len(dataset))

    # 1. Get data from dataset
    item = dataset[idx]
    x_audio = item['audio_in'].unsqueeze(0).to(device) # Input (Stimulus)
    y_real = item['target'].squeeze().cpu().numpy()    # SPICE Output
    knobs = item['knobs'].unsqueeze(0).to(device)
    s_id = item['sensor_id'].unsqueeze(0).to(device)

    # 2. Get Model Prediction
    with torch.no_grad():
        y_pred = model(x_audio, knobs, s_id)

        # Ajuste de dimensiones para modelos recurrentes (LSTM/RNN)
        # Si devuelve [B, T, 1] lo pasamos a [T]
        y_pred = y_pred.cpu().squeeze().numpy()
        if y_pred.ndim > 1: y_pred = y_pred[0]

    x_audio_np = x_audio.cpu().squeeze().numpy()

    # 3. Calculate Transfer Functions (H = FFT(output) / FFT(input))
    n_fft = 2048
    X = np.fft.rfft(x_audio_np, n=n_fft)
    Y_real = np.fft.rfft(y_real, n=n_fft)
    Y_pred = np.fft.rfft(y_pred, n=n_fft)

    # Avoid division by zero
    H_real = Y_real / (X + 1e-9)
    H_pred = Y_pred / (X + 1e-9)

    freqs = np.fft.rfftfreq(n_fft, d=1/fs)

    # 4. Plotting
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

    # Sensor mapping for the title
    sensor_names = {0: "Thermal", 1: "Photo", 2: "Mechanical", 3: "Chemical"}
    sid_val = item['sensor_id'].item()
    sensor_name = sensor_names.get(sid_val, "Unknown")

    # Magnitude Plot
    ax1.semilogx(freqs, 20*np.log10(np.abs(H_real) + 1e-9), label='SPICE Target', color='black', lw=2, alpha=0.6)
    ax1.semilogx(freqs, 20*np.log10(np.abs(H_pred) + 1e-9), label=f'RNN Prediction', color='blue', linestyle='--', lw=2)
    ax1.set_ylabel('Magnitude (dB)', fontsize=12)
    ax1.set_title(f'Frequency Response Comparison - {sensor_name} Sensor (ID: {sid_val})', fontsize=14)
    ax1.legend()
    ax1.grid(True, which="both", alpha=0.3)

    # Phase Plot
    ax2.semilogx(freqs, np.angle(H_real, deg=True), color='black', lw=2, alpha=0.6)
    ax2.semilogx(freqs, np.angle(H_pred, deg=True), color='blue', linestyle='--', lw=2)
    ax2.set_ylabel('Phase (Degrees)', fontsize=12)
    ax2.set_xlabel('Frequency (Hz)', fontsize=12)
    ax2.grid(True, which="both", alpha=0.3)

    plt.tight_layout()
    plt.show()

# --- EJECUCIÓN ---
# Usamos el modelo LSTM cargado desde Drive
if 'RNN' in dict_modelos:
    plot_comparative_bode(dict_modelos['RNN'], test_ds, device, model_name="LSTM")
else:
    print(" RNN model not found in dict_modelos.")

In [ ]:
# Respuesta de sensores

def plot_universal_results(model, test_loader, device, num_samples=4):
    model.eval()
    samples_found = {}
    # Translated sensor names for the report
    sensor_names_en = {0: "Thermal", 1: "Photo", 2: "Mechanical", 3: "Chemical"}

    # 1. Search for one example of each sensor type in the test set
    with torch.no_grad():
        for batch in test_loader:
            audio_in = batch['audio_in'].to(device)
            knobs = batch['knobs'].to(device)
            s_ids = batch['sensor_id']
            targets = batch['target'].to(device)

            # Inference
            preds = model(audio_in, knobs, s_ids.to(device))

            # Ensure predictions match target dimensions [B, 1, T]
            if preds.shape != targets.shape:
                preds = preds.transpose(1, 2) if preds.shape[1] != 1 else preds

            for i in range(len(s_ids)):
                sid = s_ids[i].item()
                if sid not in samples_found and len(samples_found) < num_samples:
                    samples_found[sid] = {
                        'target': targets[i].cpu().squeeze().numpy(),
                        'pred': preds[i].cpu().squeeze().numpy(),
                        'knobs': batch['knobs'][i].numpy(),
                        'name': sensor_names_en.get(sid, f"ID {sid}")
                    }
            if len(samples_found) == num_samples: break

    # 2. Plotting
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.flatten()

    for i, (sid, data) in enumerate(sorted(samples_found.items())):
        ax = axes[i]
        t = data['target']
        p = data['pred']

        # Local metrics for the plot
        noise = np.sum((t - p)**2)
        signal = np.sum(t**2) + 1e-7
        esr = noise / signal
        corr = np.corrcoef(t, p)[0, 1]

        # Plot Ground Truth vs Prediction
        ax.plot(t, label='SPICE (Ground Truth)', color='black', alpha=0.4, lw=2)
        ax.plot(p, label='RNN Prediction', color='red', linestyle='--', lw=1.5)

        ax.set_title(f"Sensor: {data['name']} (ID: {sid})\nESR: {esr:.6f} | Corr: {corr:.4f}", fontsize=12)
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_xlabel("Time Samples")
        ax.set_ylabel("Normalized Amplitude (V)")

    plt.suptitle("RNN Universal Model Performance Across All Sensors", fontsize=16, y=1.02)
    plt.tight_layout()

    # Save figure
    save_path = "universal_rnn_results_en.png"
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    print(f" Comparison plot saved as '{save_path}'")

# --- EXECUTION ---
# Using the TCN model loaded from your Drive dictionary
if 'RNN' in dict_modelos:
    plot_universal_results(dict_modelos['RNN'], test_loader, device)
else:
    print(" Error: 'RNN' model not found in dict_modelos. Please verify the Drive loading step.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def comparar_todos_los_sensores(lista_modelos, dataset, device):
    # Nombres en inglés para el reporte
    sensor_names = {0: "Thermal", 1: "Photo", 2: "Mechanical", 3: "Chemical"}
    colores = {'TCN': '#e74c3c', 'LSTM': '#3498db', 'RNN': '#2ecc71'}

    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    axes = axes.flatten()

    for sid in range(4):
        # 1. Buscar la primera muestra en el dataset que coincida con el ID
        idx = next(i for i, m in enumerate(dataset) if dataset[i]['sensor_id'] == sid)
        item = dataset[idx]

        audio_in = item['audio_in'].unsqueeze(0).to(device)
        knobs = item['knobs'].unsqueeze(0).to(device)
        s_id = item['sensor_id'].unsqueeze(0).to(device)
        target = item['target'].squeeze().cpu().numpy()

        ax = axes[sid]

        # Plot Ground Truth
        ax.plot(target, label='SPICE (Ground Truth)', color='black', lw=2, alpha=0.3)

        # Plot cada modelo en el diccionario
        for name, model in lista_modelos.items():
            model.eval()
            with torch.no_grad():
                pred = model(audio_in, knobs, s_id).squeeze()
                if pred.ndim > 1: pred = pred[0]
                pred_np = pred.cpu().numpy()

            ax.plot(pred_np, label=f'{name} Prediction', linestyle='--', lw=1.2, color=colores.get(name))

        ax.set_title(f"{sensor_names[sid]} Sensor (ID: {sid})", fontsize=14)
        ax.set_xlabel("Time Samples")
        ax.set_ylabel("Normalized Amplitude")
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, alpha=0.2)

    plt.suptitle("Cross-Validation across all Kaspix Sensor Types", fontsize=18, y=1.02)
    plt.tight_layout()
    plt.show()

# --- EXECUTION ---
comparar_todos_los_sensores(dict_modelos, test_ds, device)

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

def evaluar_modelos_finales(dict_modelos, loader, device):
    resultados = []

    print(" Evaluando modelos en el conjunto de prueba...")

    for name, model in dict_modelos.items():
        model.eval()
        all_esr, all_mae, all_corr = [], [], []

        with torch.no_grad():
            for batch in tqdm(loader, desc=f"Evaluando {name}", leave=False):
                audio_in = batch['audio_in'].to(device)
                knobs = batch['knobs'].to(device)
                s_id = batch['sensor_id'].to(device)
                target = batch['target'].to(device)

                # Inferencia y ajuste de dimensiones
                out = model(audio_in, knobs, s_id)
                if out.shape != target.shape:
                    out = out.transpose(1, 2) if out.shape[1] != 1 else out

                # Calcular métricas del batch
                esr, mae, corr = compute_metrics(out, target)
                all_esr.append(esr)
                all_mae.append(mae)
                all_corr.append(corr)

        # Guardar promedios
        resultados.append({
            "Modelo": name,
            "ESR (Error Ratio) ↓": np.mean(all_esr),
            "MAE (Voltios) ↓": np.mean(all_mae),
            "Correlación (Fase) ↑": np.mean(all_corr)
        })

    # Crear DataFrame
    df = pd.DataFrame(resultados)
    return df

# --- EJECUCIÓN ---
# Asegúrate de que 'dict_modelos' contenga los objetos cargados (TCN, LSTM, RNN)
tabla_comparativa = evaluar_modelos_finales(dict_modelos, test_loader, device)

# Mostrar tabla con formato
pd.options.display.float_format = '{:,.6f}'.format
display(tabla_comparativa)